***************************************************************************************

Preparing Workspace

***************************************************************************************

In [ ]:


EXPORT=False


import numpy as np
import pandas as pd
from pathlib import Path
import yaml
import plotly.express as px
from IPython.display import display
import sys


PATH_SP   = Path.home() / 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
PATH_IN = PATH_SP / 'Process Revamp' / 'Task 9. Collect new data' / 'DOF'
PATH_MAIN = PATH_SP / 'Data'
PATH_DOF = PATH_MAIN / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics'

PATH_GIT = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'DOF'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")

FILE_AREA = PATH_CONFIG0 / 'area_codes.xlsx'
FILE_YAML = PATH_CONFIG0 / 'config_indicators.yaml'

try:
    with open(FILE_YAML, 'r') as yaml_file:
        DT_CONFIG = yaml.load(yaml_file, Loader=yaml.SafeLoader)
except FileNotFoundError:
    print(f"Error: The file at {str(FILE_YAML)} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")
    


sys.path.append(str(PATH_CONFIG0))
import functions as func


def export_dof(df, indicator, geography, df_about, path):

    if 'webmapping-svr' not in str(path):
        folder = DT_CONFIG['Indicators']['Monitoring and Reporting'][indicator]['folder']
        folder = f'{indicator} {folder}'
        path = path / folder
    
    print(f"Exporting to the following location: {path}")
    print()
    
    file_out = path / f"{indicator} DOF {geography}.xlsx"
    with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
        df_about.to_excel(writer, index=False, header=False, sheet_name='About')
        df      .to_excel(writer, index=False              , sheet_name='Data' )



***************************************************************************************

Preparing Import Parameters

***************************************************************************************

In [ ]:
## Import E5 and E8 DOF data ##
list_files = [filepath for filepath in PATH_IN.iterdir()]

dt_years = {
    2000: 'Closed_E8_Full_Decade_Final_v2.xlsx'
    , 2010: 'E-8_2010_2020_by_Geo_Internet.xlsx'
    , 2020: 'E-5-2025_Geo_InternetVersion.xlsx'
}

for key, item in dt_years.items():
    file = [file for file in list_files if item in str(file)][0]
    dt_years[key] = file

print(dt_years)


# Import Area Codes
df_fips = pd.read_excel(FILE_AREA, sheet_name='CountyFIPS')
df_fips = df_fips[df_fips['STATE'] == 'CA'].dropna()

df_fips['COUNTYFP'] = df_fips['COUNTYFP'].astype(str).apply('{:0>3}'.format)
df_fips['STATEFP' ] = df_fips['STATEFP' ].astype(str).apply('{:0>2}'.format)

SACOG_counties = df_fips[df_fips['MPO'] == 'SACOG']['COUNTYNAME'].values

df_fips = df_fips.rename(columns={'COUNTYFP':'County FIPS', 'STATEFP':'State FIPS', 'COUNTYNAME':'County Name'})

print(SACOG_counties)
display(df_fips.head())


***************************************************************************************

Importing

***************************************************************************************

_Import code was last updated on Aug 7, 2024_

In [ ]:

print(); print()

# Import E5 and E8 DOF Data


# Create empty lists to store pulled data
list_df_state    = []
list_df_counties = []
list_df_cities   = []
list_df_balance  = []



# Import 1 decade at a time
# 2000-2010 data is organized differently than 2010-2024
for year, file_dof in dt_years.items():

    print(f"Importing data from the {year}'s...")

    if year == 2000:

        # Pull data using url and custom user agent
        # Drop missings, convert date field, create year field
        # Fill in county and city fields (the excel data sheet just has missings where it should be city/county)
        # Separate out and roll up population/household totals by city/county/balance
        # San francisco is special case that needs to be manually adjusted

        df = pd.read_excel(file_dof, sheet_name=1, skiprows=2)
        df = df.dropna(subset = ['Household'])
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        
        # Initialize 'County' and 'City' columns
        df['County'] = np.nan
        df['City'  ] = np.nan
        
        county_temp = None
        
        # Reset index
        df = df.reset_index(drop=True)
        
        for i in range(len(df)):
            if pd.notnull(df.loc[i, 'County / City']):
                if county_temp is not None:
                    df.loc[i, 'County'] = county_temp
                    df.loc[i, 'City'] = df.loc[i, 'County / City']
                    county_temp = None
                else:
                    county_temp = df.loc[i, 'County / City']
        
        df['County'] = df['County'].fillna(method='ffill')
        df['City'  ] = df['City'  ].fillna(method='ffill')

        # shift up and then fill the last row
        df['County'] = df['County'].shift(-1)
        df['City'  ] = df['City'  ].shift(-1)

        df['County'] = df['County'].fillna(method='ffill')
        df['City'  ] = df['City'  ].fillna(method='ffill')

        df = df.drop(columns = ['County / City'])

        df = df[df['Year'] != 2010]

        df = df.rename(columns={'Household': 'Household Population'
                                  , 'Total'  : 'Population'
                                  , 'Total.1': 'Housing Units'})
        
        df['Unoccupied'] = df['Housing Units'] - df['Occupied']
        df = df[['County', 'City', 'Date', 'Year', 'Population', 'Household Population', 'Group Quarters', 
                                  'Housing Units', 'Single'    , 'Multiple'            , 'Mobile Homes'  , 'Occupied', 'Unoccupied',
                                   'Vacancy Rate', 'Persons Per Household']].rename(columns={'Persons Per Household':'Persons per Household'})
        
        df_state = df[df['County'] == 'California'].rename(columns={'County':'State'})
        df       = df[df['County'] != 'California']
        df = df.merge(df_fips[['County Name', 'MPO']], left_on='County', right_on='County Name', how='left').drop('County Name', axis=1)
        df.loc[df['MPO'].isna(), 'MPO'] = 'Rest of CA'
        
        # Subset
        df_sf = df[df['County'] == 'San Francisco']
        df_sf.loc[:, 'City'] = 'County Total'
        df = pd.concat([df, df_sf])
        df['City'] = df['City'].str.replace(' City', '')
        df = df[~df['Year'].isna()]
        df_cities   = df[~df['City'].isin(['County Total', 'Incorporated', 'Balance of County'])].reset_index(drop=True)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop=True).drop('City', axis=1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop=True).drop('City', axis=1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )

    else:

        # Pull data using url and custom user agent
        # Drop missings, convert date field, create year field
        # Separate out and roll up population/household totals by city/county/balance
        # San francisco is special case that needs to be manually adjusted

        df = pd.read_excel(file_dof, sheet_name=1, skiprows=2)
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        if year == 2010:
            df = df[df['Year'] != 2020]

        df = df.rename(columns={'Household': 'Household Population'
                                , 'Total'  : 'Population'
                                , 'Total.1': 'Housing Units'})
        if year == 2010:
            cols_to_int = ['Population', 'Household Population', 'Group Quarters', 'Housing Units', 'Single Detached', 'Single Attached',
                            'Two to Four', 'Five Plus', 'Mobile Homes', 'Occupied', 'Vacancy Rate', 'Persons per Household']
            df[cols_to_int] = df[cols_to_int].apply(pd.to_numeric, errors='coerce')
            df = df.dropna()

        df['Unoccupied'] = df['Housing Units'  ] - df['Occupied'       ]
        df['Single'    ] = df['Single Attached'] + df['Single Detached']
        df['Multiple'  ] = df['Two to Four'    ] + df['Five Plus'      ]

        df = df[['County', 'City', 'Date', 'Year', 'Population', 'Household Population', 'Group Quarters', 
                                    'Housing Units', 'Single'    , 'Single Detached'     , 'Single Attached', 'Multiple',
                                    'Two to Four', 'Five Plus' , 'Mobile Homes'        ,
                                        'Occupied', 'Unoccupied', 'Vacancy Rate'        , 'Persons per Household']]

        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        df = df.merge(df_fips[['County Name', 'MPO']], left_on='County', right_on='County Name', how='left').drop('County Name', axis=1)
        df.loc[df['MPO'].isna(), 'MPO'] = 'Rest of CA'

        # Subset
        if year == 2020:
            df_sf = df[df['County'] == 'San Francisco']
            df_sf.loc[:, 'City'] = 'San Francisco'
            df = pd.concat([df, df_sf])

        if year == 2010:
            df_sf = df[df['County'] == 'San Francisco']
            df_sf.loc[:, 'City'] = 'County Total'
            df = pd.concat([df, df_sf])
        df = df[~df['City'].isna()]
        df['City'] = df['City'].str.replace(' City', '')
        df = df[~df['Year'].isna()]
        df_cities   = df[~df['City'].isin(['County Total', 'Incorporated', 'Balance of County'])].reset_index(drop=True)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop=True).drop('City', axis=1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop=True).drop('City', axis=1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )
            

df_state    = pd.concat(list_df_state   )
df_counties = pd.concat(list_df_counties)
df_cities   = pd.concat(list_df_cities  )
df_balance  = pd.concat(list_df_balance )

df_state    = df_state   .sort_values(['City'  , 'Year'], ascending=[True, False]).rename(columns={'City':'Jurisdiction'})
df_counties = df_counties.sort_values(['County', 'Year'], ascending=[True, False])
df_cities   = df_cities  .sort_values(['City'  , 'Year'], ascending=[True, False]).rename(columns={'City':'Jurisdiction'})
df_balance  = df_balance .sort_values(['County', 'Year'], ascending=[True, False])


df_counties = df_counties.set_index(['MPO', 'County'                ]).reset_index()
df_cities   = df_cities  .set_index(['MPO', 'County', 'Jurisdiction']).reset_index()
df_balance  = df_balance .set_index(['MPO', 'County'                ]).reset_index()


print(); print()
print('By State')
print(df_state.head())
print('By Counties')
print(df_counties.head())
print('By Cities')
print(df_cities.head())
print('By Balance')
print(df_balance.head())

# Mariposa, Alpine, Trinity not included in the "Cities" df because they don't have any jurisdictions


# Export raw data in full
paths = [PATH_DOF, PATH_SERVER]
# paths = [path_out_sp]



print(); print()


# All of CA
df_cities2  = df_cities .copy()
df_balance2 = df_balance.copy()
df_counties2 = df_counties.copy()

df_counties2 = df_counties2[df_counties2['County'].isin(['Trinity', 'Mariposa', 'Alpine'])]

df_counties2['Jurisdiction'] = 'Unincorporated'
df_balance2['Jurisdiction'] = 'Unincorporated'
df_cities3 = pd.concat([df_cities2, df_balance2, df_counties2])


df_cities3 = df_cities3.sort_values(['County', 'Jurisdiction', 'Year'], ascending=[True, True, False])
df_cities3 = df_cities3.reset_index(drop=True)


if EXPORT:
    file_out = PATH_DOF / 'DOF_E5_and_E8_Jurisdictions.xlsx'
    df_cities3.to_excel(file_out, index=False)
    
print(df_cities3.Year.unique())
display(df_cities3.head())



***************************************************************************************

Organize Indicators

***************************************************************************************

In [ ]:


# Sample we are pulling from
sample_type = 'DOF'
year_start  = df_cities.Year.min()
year_end    = df_cities.Year.max()



In [ ]:
print(); print()


## Pop_1 -------------------------------------------------------------------------------------------------


indicator = 'Pop_1'

# Import about table
geography = 'Counties'
df_about = func.write_about(sample_type, indicator, year_start, year_end)
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography
print("About page documentation table:")
display(df_about)
print()

# Organize indicator
df_pop1 = df_counties[df_counties['County'].isin(SACOG_counties)]
df_pop1 = df_pop1[['County', 'Year', 'Population', 'Household Population']].drop_duplicates()


# View
print('Indicator table:')
display(df_pop1.head())


# Export
if EXPORT:
    for path in paths:
        export_dof(df_pop1, indicator, geography, df_about, path)


print()
print("Successfully exported!   \m/( -_- )")



In [ ]:
print(); print()


df_plot = df_pop1.copy()

x = 'Year'
y = 'Population'
color = 'County'
labels = 'County'

fig = px.line(df_plot, x=x, y=y, color=color, markers=True, labels=labels)
fig.update_layout(title = 'Population by County')

fig.show()



In [ ]:
print(); print()

## Pop_2 -------------------------------------------------------------------------------------------------


indicator = 'Pop_2'


## Jurisdictions ---

# Import about table
geography = 'Jurisdictions'
df_about = func.write_about(sample_type, indicator, year_start, year_end)
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography


df_pop2 = df_cities[df_cities['County'].isin(SACOG_counties)]
df_pop2_balance = df_balance[df_balance['County'].isin(SACOG_counties)]
df_pop2_balance['Jurisdiction'] = 'Unincorporated'

df_pop2 = pd.concat([df_pop2, df_pop2_balance])  
df_pop2 = df_pop2[['County', 'Jurisdiction', 'Year', 'Population', 'Household Population']]
df_pop2 = df_pop2[df_pop2['Jurisdiction'] != 'Incorporated']
df_pop2 = df_pop2.sort_values(['Jurisdiction', 'Year'], ascending = [True, True])
df_pop2['Population_GR'          ] = df_pop2['Population'          ].pct_change()*100
df_pop2['Household Population_GR'] = df_pop2['Household Population'].pct_change()*100
df_pop2.loc[df_pop2['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop2.loc[df_pop2['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop2 = df_pop2.sort_values(['County', 'Jurisdiction', 'Year'], ascending = [True, True, False])

# Export
if EXPORT:
    for path in paths:
        export_dof(df_pop2, indicator, geography, df_about, path)



## Counties ---

# Import about table
geography = 'Counties'
df_about = func.write_about(sample_type, indicator, year_start, year_end)
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography

df_pop2 = df_counties[df_counties['County'].isin(SACOG_counties)]
df_pop2 = df_pop2[['County', 'Year', 'Population', 'Household Population']]
df_pop2 = df_pop2[df_pop2['County'] != 'Incorporated']
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, True])
df_pop2['Population_GR'          ] = df_pop2['Population'          ].pct_change()*100
df_pop2['Household Population_GR'] = df_pop2['Household Population'].pct_change()*100
df_pop2.loc[df_pop2['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop2.loc[df_pop2['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, False])

# Export
if EXPORT:
    for path in paths:
        export_dof(df_pop2, indicator, geography, df_about, path)



# 5 year time lapses
print("About page documentation table:")
df_about.loc[df_about['Indicator'] == 'Year(s)', indicator] = f"{year_start}-{year_end}, 5-Year Time Series"
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography

display(df_about)
print('')

df_pop2_5 = df_counties[df_counties['County'].isin(SACOG_counties)]
df_pop2_5 = df_pop2_5[df_pop2_5['Year'].isin([2000, 2005, 2010, 2015, 2020, 2024])]
df_pop2_5 = df_pop2_5[['County', 'Year', 'Population']]
df_pop2_5 = df_pop2_5.sort_values(['County', 'Year'], ascending = [True, True])
df_pop2_5['Population_Diff'] = df_pop2_5.groupby('County')['Population'].diff()
df_pop2_5['Year_Diff'] = df_pop2_5.groupby('County')['Year'].diff()
df_pop2_5['Year_Diff'].fillna(1, inplace=True)  # Assuming the difference of one year when NaN, adjust if needed
df_pop2_5['Population_Diff'].fillna(0, inplace=True)

df_pop2_5['Previous_POP'] = df_pop2_5.groupby('County')['Population'].shift(1)
df_pop2_5['Result'] = round(((df_pop2_5['Population_Diff']) / df_pop2_5['Previous_POP']) * 100 / df_pop2_5['Year_Diff'], 1)
df_pop2_5 = df_pop2_5[~df_pop2_5['Result'].isna()]
df_pop2_5 = df_pop2_5.sort_values(['County', 'Year'], ascending = [True, False])


df_pop2_5 = df_pop2_5.pivot_table(index = ['County']
                                 , columns = 'Year'
                                 , values = 'Result').reset_index()

df_pop2_5.columns = ['County', '2000-2005', '2005-2010', '2010-2015', '2015-2020', '2020-2025']

df_pop2_5['2000-2005'] = df_pop2_5['2000-2005'].astype('str').apply(lambda x: str(x)+'%')
df_pop2_5['2005-2010'] = df_pop2_5['2005-2010'].astype('str').apply(lambda x: str(x)+'%')
df_pop2_5['2010-2015'] = df_pop2_5['2010-2015'].astype('str').apply(lambda x: str(x)+'%')
df_pop2_5['2015-2020'] = df_pop2_5['2015-2020'].astype('str').apply(lambda x: str(x)+'%')
df_pop2_5['2020-2025'] = df_pop2_5['2020-2025'].astype('str').apply(lambda x: str(x)+'%')


print("Indicator Table: ")
display(df_pop2_5.head())

# Export
geography = 'Counties_5 Year Time Series'
if EXPORT:
    for path in paths:
        export_dof(df_pop2_5, indicator, geography, df_about, path)

print()
print("Successfully exported!   \m/( -_- )")

print(); print()


In [ ]:
print(); print()


df_plot = df_pop2.copy()
df_plot['Population_GR'] = round(df_plot['Population_GR']*100, 1)

x = 'Year'
y = 'Population_GR'
color = 'County'
labels = 'County'

fig = px.line(df_plot, x=x, y=y, color=color, markers=True, labels=labels)
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'black')
fig.update_layout(title = 'Population Growth Rate by County (%)')

fig.show()



In [ ]:
print(); print()


## Pop_5 -------------------------------------------------------------------------------------------------


indicator = 'Pop_5'


## Jurisdiction ---

# Import about table
geography = 'Jurisdictions'
df_about = func.write_about(sample_type, indicator, year_start, year_end)
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography

df_pop5 = df_cities[df_cities['County'].isin(SACOG_counties)]
df_pop5_balance = df_balance[df_balance['County'].isin(SACOG_counties)]
df_pop5_balance['Jurisdiction'] = 'Unincorporated'
df_pop5 = pd.concat([df_pop5, df_pop5_balance])  

df_pop5 = df_pop5[['MPO', 'County', 'Jurisdiction', 'Year', 'Population', 'Household Population']]
df_pop5 = df_pop5[df_pop5['Jurisdiction'] != 'Incorporated']
df_pop5 = df_pop5.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, True])
df_pop5['Population_GR'          ] = df_pop5['Population'          ].pct_change()
df_pop5['Household Population_GR'] = df_pop5['Household Population'].pct_change()
df_pop5.loc[df_pop5['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop5.loc[df_pop5['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop5 = df_pop5.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, False])

# Export
if EXPORT:
    for path in paths:
        export_dof(df_pop5, indicator, geography, df_about, path)



# Counties ---

# Import about table
geography = 'Counties'
df_about = func.write_about(sample_type, indicator, year_start, year_end)
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography

df_pop5 = df_counties[['MPO', 'County', 'Year', 'Population', 'Household Population']]
df_pop5 = df_pop5[df_pop5['County'] != 'Incorporated']
df_pop5 = df_pop5.sort_values(['County', 'Year'], ascending = [True, True])
df_pop5['Population_GR'          ] = df_pop5['Population'          ].pct_change()
df_pop5['Household Population_GR'] = df_pop5['Household Population'].pct_change()
df_pop5.loc[df_pop5['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop5.loc[df_pop5['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop5 = df_pop5.sort_values(['MPO', 'County', 'Year'], ascending = [True, True, False])

# Export
if EXPORT:
    for path in paths:
        export_dof(df_pop5, indicator, geography, df_about, path)



## MPO ---

# Import about table
geography = 'MPO'
df_about = func.write_about(sample_type, indicator, year_start, year_end)
df_about.loc[df_about['Indicator'] == 'Geography', indicator] = geography

display(df_about)
print()

df_pop5 = df_counties[df_counties['County'] != 'Incorporated']
df_pop5 = df_pop5[['MPO', 'Year', 'Population', 'Household Population']]
df_pop5 = df_pop5.groupby(['MPO', 'Year'], as_index = False).agg(sum)
df_pop5 = df_pop5.sort_values(['MPO', 'Year'], ascending = [True, True])
df_pop5['Population_GR'          ] = df_pop5['Population'          ].pct_change()
df_pop5['Household Population_GR'] = df_pop5['Household Population'].pct_change()
df_pop5.loc[df_pop5['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop5.loc[df_pop5['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop5 = df_pop5.sort_values(['MPO', 'Year'], ascending = [True, False])


# View
print("Indicator Table: ")
display(df_pop5.head())

# Export
if EXPORT:
    for path in paths:
        export_dof(df_pop5, indicator, geography, df_about, path)


print()
print("Successfully exported!   \m/( -_- )")

print(); print()


In [ ]:
print(); print()


## Just by MPO ##
df_plot = df_pop5.copy()
df_plot['Population_GR'] = round(df_plot['Population_GR']*100, 1)


x = 'Year'
y = 'Population_GR'
color = 'MPO'


fig = px.line(df_plot, x=x, y=y, color=color, markers=True)

fig.add_hline(y = 0, line_dash = 'dash', line_color = 'black')
fig.update_layout(title = 'Population Growth Rate by MPO (%)')

fig.show()

